In [1]:
from typing import Callable
from socceraction.spadl.utils import add_names
import numpy as np
import pandas as pd
import torch
import itertools
import timeit
from rich.progress import track
from torch.utils.data import DataLoader, Subset, random_split
from unxpass.components import pass_selection, pass_success, pass_value, pass_value_custom
from unxpass.datasets import PassesDataset
def convert_pass_coords(pass_selection_surface, x_t, y_t):
    #edited to convert selection surface coords to real x and y
    p_odds = pass_selection_surface[y_t, x_t]
    y_dim, x_dim = pass_selection_surface.shape
    y_t = y_t / y_dim * 68 + 68 / y_dim / 2
    x_t = x_t / x_dim * 105 + 105 / x_dim / 2
    
    return x_t, y_t, p_odds
def convert_x(x_t, x_dim):
    return x_t / x_dim * 105 + 105 / x_dim / 2
def convert_y(y_t, y_dim):
    return y_t / y_dim * 68 + 68 / y_dim / 2

class LocationPredictions:
    def __init__(
        self,
        pass_selection_component: pass_selection.SoccerMapComponent,
        pass_success_component: pass_success.SoccerMapComponent,
        pass_value_success_offensive_component: pass_value_custom.SoccerMapComponent,
        pass_value_success_defensive_component: pass_value_custom.SoccerMapComponent,
        pass_value_fail_offensive_component: pass_value_custom.SoccerMapComponent,
        pass_value_fail_defensive_component: pass_value_custom.SoccerMapComponent
    ):
        self.pass_value_success_offensive_component = pass_value_success_offensive_component
        self.pass_value_success_defensive_component = pass_value_success_defensive_component
        self.pass_selection_component = pass_selection_component
        self.pass_success_component = pass_success_component
        self.pass_value_fail_offensive_component = pass_value_fail_offensive_component
        self.pass_value_fail_defensive_component = pass_value_fail_defensive_component
    def setup():
        pass_selection_surface = self.pass_selection_component.predict_surface(dataset, db = db, model_name = "sel")
        pass_success_surface = self.pass_success_component.predict_surface(dataset,db = db,model_name = "val")
        pass_value_surface_offensive_success = self.pass_value_success_offensive_component.predict_surface(dataset, db = db,model_name = "val")
        pass_value_surface_offensive_fail = self.pass_value_fail_offensive_component.predict_surface(dataset, db = db,model_name = "val")
        pass_value_surface_defensive_success = self.pass_value_success_defensive_component.predict_surface(dataset, db = db,model_name = "val")
        pass_value_surface_defensive_fail = self.pass_value_fail_defensive_component.predict_surface(dataset, db = db,model_name = "val")
        
    def rate_all_games(self, db, dataset, summarize = True, custom_pass = None):
        print("Generating Surfaces:")
        pass_selection_surface = self.pass_selection_component.predict_surface(dataset, db = db, model_name = "sel")
        pass_success_surface = self.pass_success_component.predict_surface(dataset,db = db,model_name = "val")
        pass_value_surface_offensive_success = self.pass_value_success_offensive_component.predict_surface(dataset, db = db,model_name = "val")
        pass_value_surface_offensive_fail = self.pass_value_fail_offensive_component.predict_surface(dataset, db = db,model_name = "val")
        pass_value_surface_defensive_success = self.pass_value_success_defensive_component.predict_surface(dataset, db = db,model_name = "val")
        pass_value_surface_defensive_fail = self.pass_value_fail_defensive_component.predict_surface(dataset, db = db,model_name = "val")
        print("Finished.")
        alldf = []
        sels = self.pass_selection_component.predict(dataset,model_name = "sel")#ensuring that the actual pass made is in the dataset
        value_fail = self.pass_value_fail_offensive_component.predict(dataset, model_name = "val") - self.pass_value_fail_defensive_component.predict(dataset, model_name = "val")
        value_success = self.pass_value_success_offensive_component.predict(dataset, model_name = "val") - self.pass_value_success_defensive_component.predict(dataset, model_name = "val")
        success = self.pass_success_component.predict(dataset, model_name = "val")
        all_ratings = pd.concat([sels, success, value_fail, value_success], axis=1).rename(columns = {0:"selection_probability", 1: "success_probability", 2:"value_fail", 3:"value_success"})
        for game in pass_selection_surface:
            game_preds = all_ratings.loc[game]
            ending_coords = db.actions(game_id = game)[["end_x", "end_y"]].loc[game]
            game_ogs = pd.concat([game_preds,ending_coords], axis = 1).dropna()
            game_df_actions = db.actions(game_id = game)
            game_selection = pass_selection_surface[game][next(iter(pass_selection_surface[game]))]
            x_lim, y_lim = game_selection.shape
            coords = list(itertools.product(range(0,x_lim), range(0,y_lim)))
            coords_array = np.array(coords)
            input_df = pd.DataFrame()
            input_df["coord_x"] = coords_array[:, 0]
            input_df["coord_y"] = coords_array[:, 1]
            coord_x = input_df["coord_x"].values
            coord_y = input_df["coord_y"].values
            for action in pass_selection_surface[game]:
                if custom_pass is not None:
                    game = custom_pass["game_id"]
                    action = custom_pass["action_id"]
                print(game, action)
                allpasses = pd.DataFrame([game_df_actions.loc[(game, action)]] * len(coords)).reset_index(drop = True)
                input_df_act = pd.concat([allpasses, input_df], axis = 1)
                true_ends = game_df_actions.loc[(game,action)][["end_x","end_y"]].reset_index(drop = True)#this is really slow...
                input_df_act["end_x"] = coord_x / x_lim * 105 + 105 / x_lim / 2
                input_df_act["end_y"] = coord_y / y_lim * 68 + 68 / y_lim / 2
                #metrics = self.rate_optimized(game, action, pass_selection_surface, pass_success_surface, pass_value_surface_offensive_success, pass_value_surface_defensive_success,pass_value_surface_offensive_fail, pass_value_surface_defensive_fail, db, game_df_actions)
                return input_df_act, game, action, pass_selection_surface, pass_success_surface, pass_value_surface_offensive_success, pass_value_surface_defensive_success,pass_value_surface_offensive_fail, pass_value_surface_defensive_fail, db, game_df_actions
                metrics = self.rate(input_df_act, game, action, pass_selection_surface, pass_success_surface, pass_value_surface_offensive_success, pass_value_surface_defensive_success,pass_value_surface_offensive_fail, pass_value_surface_defensive_fail, db, game_df_actions)
                #print(metrics.columns) 
                #print(true_ends)               
                metrics["Dist_From_True"] = (metrics["end_x"] - true_ends[0])**2 + (metrics["end_y"] - true_ends[1])**2
                closest_idx = np.where(metrics["Dist_From_True"] == min(metrics["Dist_From_True"]))[0]#getting closest pass to actual pass and then replacing that one with the original pass
                metrics = metrics.reset_index(drop = True)#is the slowest here?
                cols = ["end_x", "end_y", "selection_probability", "success_probability", "value_success", "value_fail"]
                metrics["True_Location"] = 0
                for col in cols:
                    metrics[col][closest_idx[0]] = game_ogs[col].loc[action]
                metrics["selection_probability"] = np.float64(metrics["selection_probability"])
                metrics["True_Location"][closest_idx[0]] = 1
                metrics["expected_utility"] = (metrics["success_probability"] * metrics["value_success"]) + ((1 - metrics["success_probability"]) * metrics["value_fail"])
                metrics["evaluation_criterion"] = metrics["expected_utility"] - sum(metrics["selection_probability"] * metrics["expected_utility"])
                metrics["selection_criterion"] = sum(
                metrics["selection_probability"] * (metrics["evaluation_criterion"])**2
                )
                metrics = metrics.drop(columns = ["Dist_From_True"])
                if summarize:
                    metrics = metrics[metrics["True_Location"] == 1]
                    metrics = metrics[["original_event_id", "game_id", "action_id", "start_x","start_y", "end_x", "end_y", "result_id", "selection_criterion", "evaluation_criterion"]]
                if custom_pass is not None:
                    return metrics
                alldf.append(metrics)
            
        combined = pd.concat(alldf)
        return combined

    def rate(self, input_df, game, action, selection, success, value_success_off, value_success_def, value_fail_off, value_fail_def, db, game_df_actions):
        start_time = timeit.default_timer()
        game_selection = selection[game][action]
        game_success = success[game][action]
        game_value_success_off = value_success_off[game][action]
        game_value_success_def = value_success_def[game][action]
        game_value_fail_off = value_fail_off[game][action]
        game_value_fail_def = value_fail_def[game][action]
        #x_lim, y_lim = game_selection.shape
        #coords = list(itertools.product(range(0,x_lim), range(0,y_lim)))
        #test_db = game_df_actions
        elapsed = timeit.default_timer() - start_time
        start_time = timeit.default_timer()
        df_override = input_df #pd.DataFrame([test_db.loc[(game, action)]] * len(coords))
        df_override["game_id"] = game
        df_override["action_id"] = action
        elapsed = timeit.default_timer() - start_time
        start_time = timeit.default_timer()
        #df_override["coord_x"] = coords_array[:, 0]
        #df_override["coord_y"] = coords_array[:, 1]
        elapsed = timeit.default_timer() - start_time
        start_time = timeit.default_timer()
        #df_override["end_x"] = convert_x(df_override["coord_x"], x_lim)
        #df_override["end_y"] = convert_y(df_override["coord_y"], y_lim)
        elapsed = timeit.default_timer() - start_time
        start_time = timeit.default_timer()
        df_override["selection_probability"] = game_selection[df_override["coord_x"], df_override["coord_y"]]
        df_override["success_probability"] = game_success[df_override["coord_x"], df_override["coord_y"]]
        df_override["value_success_off"] = game_value_success_off[df_override["coord_x"], df_override["coord_y"]]
        df_override["value_fail_off"] = game_value_fail_off[df_override["coord_x"], df_override["coord_y"]]
        df_override["value_success_def"] = game_value_success_def[df_override["coord_x"], df_override["coord_y"]]
        df_override["value_fail_def"] = game_value_fail_def[df_override["coord_x"], df_override["coord_y"]]
        df_override["value_success"] = df_override["value_success_off"] - df_override["value_success_def"]
        df_override["value_fail"] = df_override["value_fail_off"] - df_override["value_fail_def"]
        elapsed = timeit.default_timer() - start_time
        start_time = timeit.default_timer()
        return df_override



In [21]:

from pathlib import Path
from functools import partial
import pandas as pd
import matplotlib.pyplot as plt
pd.options.mode.chained_assignment = None

import numpy as np
import mlflow
from scipy.ndimage import zoom

import warnings
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
from unxpass.databases import SQLiteDatabase
from unxpass.datasets import PassesDataset, CompletedPassesDataset, FailedPassesDataset
from unxpass.components import pass_selection, pass_value, pass_success,pass_value_custom
from unxpass.components.utils import load_model
from unxpass.visualization import plot_action
STORES_FP = Path("../stores")

db = SQLiteDatabase(STORES_FP / "test_50.sql")

dataset = partial(PassesDataset, path=STORES_FP / "datasets" / "custom" / "dataset_subset_1")
plt_settings = {"cmap": "magma", "vmin": 0, "vmax": 1, "interpolation": "bilinear"}

model_pass_selection = pass_selection.SoccerMapComponent(
    model=mlflow.pytorch.load_model(
        'runs:/04d45112c139473590b5049cb3797d0d/model', map_location='cpu'
        #'runs:/788ec5a232af46e59ac984d50ecfc1d5/model', map_location='cpu'
    )
)
#model_pass_selection.test(dataset_test)
#model_pass_success = load_model('runs:/f977aaf2f5a0497cb51f5e730ae64609/component')
model_pass_success = pass_success.SoccerMapComponent(
    model=mlflow.pytorch.load_model(
        'runs:/78b7ab86dc864858b8814fe811b8796a/model', map_location='cpu'
        #'runs:/788ec5a232af46e59ac984d50ecfc1d5/model', map_location='cpu'
    )
)

#
model_pass_value_success_offensive = pass_value_custom.SoccerMapComponent(#good
    model=mlflow.pytorch.load_model(
        'runs:/ec44b8ef79944b10a0d87d13949a1fd3/model', map_location='cpu'
        #'runs:/788ec5a232af46e59ac984d50ecfc1d5/model', map_location='cpu'
    )# 7 channel model
)

model_pass_value_success_defensive = pass_value_custom.SoccerMapComponent(#good
    model=mlflow.pytorch.load_model(
        'runs:/fc10352d1c4948b4ac556443e4de575a/model', map_location='cpu'
        #'runs:/788ec5a232af46e59ac984d50ecfc1d5/model', map_location='cpu'
    ), offensive = False#rest are 9 channel models
)

model_pass_value_fail_offensive = pass_value_custom.SoccerMapComponent(#ntbc
    model=mlflow.pytorch.load_model(
        'runs:/a7eb2344aa414ef582bbc06dcea6b00e/model', map_location='cpu'
        #'runs:/788ec5a232af46e59ac984d50ecfc1d5/model', map_location='cpu'
    )
)

model_pass_value_fail_defensive = pass_value_custom.SoccerMapComponent(#ntbc
    model=mlflow.pytorch.load_model(
        'runs:/28946631a0b54fe89dc83ca129815b4f/model', map_location='cpu'
        #'runs:/788ec5a232af46e59ac984d50ecfc1d5/model', map_location='cpu'
    ), offensive = False
)

#model_pass_success.test(dataset_test)
#model_pass_value.test(dataset_test)
#options = pd.read_parquet("/home/lz80/un-xPass/stores/datasets/euro2020/x_pass_options.parquet")
#true_picked = pd.read_parquet("/home/lz80/un-xPass/stores/datasets/euro2020/y_receiver.parquet")

# %%
rater = LocationPredictions(
    pass_selection_component=model_pass_selection,
    pass_success_component=model_pass_success,
    pass_value_success_offensive_component=model_pass_value_success_offensive,
    pass_value_fail_offensive_component = model_pass_value_fail_offensive,
    pass_value_success_defensive_component=model_pass_value_success_defensive,
    pass_value_fail_defensive_component = model_pass_value_fail_defensive,
)
#test = rater.rate_all_games(db, dataset_test, summarize = True)#this will take a bit, might want to make this more efficient in the future

In [22]:
pass_selection_surface = model_pass_selection.predict_surface(dataset, db = db, model_name = "sel")
pass_success_surface = model_pass_success.predict_surface(dataset,db = db,model_name = "val")
pass_value_surface_offensive_success = model_pass_value_success_offensive.predict_surface(dataset, db = db,model_name = "val")
pass_value_surface_offensive_fail = model_pass_value_fail_offensive.predict_surface(dataset, db = db,model_name = "val")
pass_value_surface_defensive_success = model_pass_value_success_defensive.predict_surface(dataset, db = db,model_name = "val")
pass_value_surface_defensive_fail = model_pass_value_fail_defensive.predict_surface(dataset, db = db,model_name = "val")
alldf = []
sels = model_pass_selection.predict(dataset,model_name = "sel")#ensuring that the actual pass made is in the dataset
value_fail = model_pass_value_fail_offensive.predict(dataset, model_name = "val") - model_pass_value_fail_defensive.predict(dataset, model_name = "val")
value_success = model_pass_value_success_offensive.predict(dataset, model_name = "val") - model_pass_value_success_defensive.predict(dataset, model_name = "val")
success = model_pass_success.predict(dataset, model_name = "val")

sel


[10/07/24 19:35:21] INFO     Loading dataset from ../stores/datasets/custom/dataset_subset_1        ]8;id=962046;file:///home/lz80/un-xPass/.venv/lib/python3.10/site-packages/lightning_utilities/core/rank_zero.py\rank_zero.py]8;;\:]8;id=263301;file:///home/lz80/un-xPass/.venv/lib/python3.10/site-packages/lightning_utilities/core/rank_zero.py#32\32]8;;\

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


Predicting: 0it [00:00, ?it/s]

Creating Output...
1/1092
2/1092
3/1092
4/1092
5/1092
6/1092
7/1092
8/1092
9/1092
10/1092
11/1092
12/1092
13/1092
14/1092
15/1092
16/1092
17/1092
18/1092
19/1092
20/1092
21/1092
22/1092
23/1092
24/1092
25/1092
26/1092
27/1092
28/1092
29/1092
30/1092
31/1092
32/1092
33/1092
34/1092
35/1092
36/1092
37/1092
38/1092
39/1092
40/1092
41/1092
42/1092
43/1092
44/1092
45/1092
46/1092
47/1092
48/1092
49/1092
50/1092
51/1092
52/1092
53/1092
54/1092
55/1092
56/1092
57/1092
58/1092
59/1092
60/1092
61/1092
62/1092
63/1092
64/1092
65/1092
66/1092
67/1092
68/1092
69/1092
70/1092
71/1092
72/1092
73/1092
74/1092
75/1092
76/1092
77/1092
78/1092
79/1092
80/1092
81/1092
82/1092
83/1092
84/1092
85/1092
86/1092
87/1092
88/1092
89/1092
90/1092
91/1092
92/1092
93/1092
94/1092
95/1092
96/1092
97/1092
98/1092
99/1092
100/1092
101/1092
102/1092
103/1092
104/1092
105/1092
106/1092
107/1092
108/1092
109/1092
110/1092
111/1092
112/1092
113/1092
114/1092
115/1092
116/1092
117/1092
118/1092
119/1092
120/1092
121/1092


...

[10/07/24 19:40:14] INFO     Loading dataset from ../stores/datasets/custom/dataset_subset_1        ]8;id=596639;file:///home/lz80/un-xPass/.venv/lib/python3.10/site-packages/lightning_utilities/core/rank_zero.py\rank_zero.py]8;;\:]8;id=310133;file:///home/lz80/un-xPass/.venv/lib/python3.10/site-packages/lightning_utilities/core/rank_zero.py#32\32]8;;\

Output()

val


[10/07/24 19:40:32] INFO     Loading dataset from ../stores/datasets/custom/dataset_subset_1        ]8;id=157169;file:///home/lz80/un-xPass/.venv/lib/python3.10/site-packages/lightning_utilities/core/rank_zero.py\rank_zero.py]8;;\:]8;id=59435;file:///home/lz80/un-xPass/.venv/lib/python3.10/site-packages/lightning_utilities/core/rank_zero.py#32\32]8;;\

Output()

In [35]:
def rate(input_df, game, action, selection, success, value_success_off, value_success_def, value_fail_off, value_fail_def, db, game_df_actions):
    start_time = timeit.default_timer()
    start_time_og = timeit.default_timer()
    game_selection = selection[game][action]
    game_success = success[game][action]
    game_value_success_off = value_success_off[game][action]
    game_value_success_def = value_success_def[game][action]
    game_value_fail_off = value_fail_off[game][action]
    game_value_fail_def = value_fail_def[game][action]
    #x_lim, y_lim = game_selection.shape
    #coords = list(itertools.product(range(0,x_lim), range(0,y_lim)))
    #test_db = game_df_actions
    elapsed = timeit.default_timer() - start_time
    #print(elapsed)
    start_time = timeit.default_timer()
    df_override = input_df #pd.DataFrame([test_db.loc[(game, action)]] * len(coords))
    df_override["game_id"] = game
    df_override["action_id"] = action
    elapsed = timeit.default_timer() - start_time
    start_time = timeit.default_timer()
    #df_override["coord_x"] = coords_array[:, 0]
    #df_override["coord_y"] = coords_array[:, 1]
    
    elapsed = timeit.default_timer() - start_time
    #print(elapsed)
    start_time = timeit.default_timer()
    #df_override["end_x"] = convert_x(df_override["coord_x"], x_lim)
    #df_override["end_y"] = convert_y(df_override["coord_y"], y_lim)
    elapsed = timeit.default_timer() - start_time
    #print(elapsed)
    start_time = timeit.default_timer()
    df_override["selection_probability"] = game_selection[df_override["coord_x"], df_override["coord_y"]]
    df_override["success_probability"] = game_success[df_override["coord_x"], df_override["coord_y"]]
    df_override["value_success_off"] = game_value_success_off[df_override["coord_x"], df_override["coord_y"]]
    df_override["value_fail_off"] = game_value_fail_off[df_override["coord_x"], df_override["coord_y"]]
    df_override["value_success_def"] = game_value_success_def[df_override["coord_x"], df_override["coord_y"]]
    df_override["value_fail_def"] = game_value_fail_def[df_override["coord_x"], df_override["coord_y"]]
    df_override["value_success"] = df_override["value_success_off"] - df_override["value_success_def"]
    df_override["value_fail"] = df_override["value_fail_off"] - df_override["value_fail_def"]
    elapsed = timeit.default_timer() - start_time
    elapsed_og = timeit.default_timer() - start_time_og
    start_time = timeit.default_timer()
    print(elapsed_og)
    return df_override


In [36]:
all_ratings = pd.concat([sels, success, value_fail, value_success], axis=1).rename(columns = {0:"selection_probability", 1: "success_probability", 2:"value_fail", 3:"value_success"})
for game in pass_selection_surface:
    start_time = timeit.default_timer()
    game_preds = all_ratings.loc[game]
    ending_coords = db.actions(game_id = game)[["end_x", "end_y"]].loc[game]
    game_ogs = pd.concat([game_preds,ending_coords], axis = 1).dropna()
    game_df_actions = db.actions(game_id = game)
    game_selection = pass_selection_surface[game][next(iter(pass_selection_surface[game]))]
    x_lim, y_lim = game_selection.shape
    coords = list(itertools.product(range(0,x_lim), range(0,y_lim)))
    coords_array = np.array(coords)
    input_df = pd.DataFrame()
    input_df["coord_x"] = coords_array[:, 0]
    input_df["coord_y"] = coords_array[:, 1]
    elapsed = timeit.default_timer() - start_time
    #print(f"1:{elapsed}")
    start_time = timeit.default_timer()
    coord_x = input_df["coord_x"].values
    coord_y = input_df["coord_y"].values
    for action in pass_selection_surface[game]:

        #if custom_pass is not None:
        #    game = custom_pass["game_id"]
        #    action = custom_pass["action_id"]
        print(game, action)
        
        elapsed = timeit.default_timer() - start_time
        #print(f"2:{elapsed}")
        start_time = timeit.default_timer()
        row = game_df_actions.loc[(game, action)].values
        #allpasses = pd.DataFrame([game_df_actions.loc[(game, action)]] * len(coords)).reset_index(drop = True)
        allpasses = pd.DataFrame(np.tile(row, (len(coords), 1)), columns=game_df_actions.columns).reset_index(drop=True)
        elapsed = timeit.default_timer() - start_time
        #print(f"2.25:{elapsed}")
        start_time = timeit.default_timer()
        true_ends = game_df_actions.loc[(game,action)][["end_x","end_y"]].reset_index(drop = True)
        input_df_act = pd.concat([allpasses, input_df], axis = 1)
        true_ends = game_df_actions.loc[(game,action)][["end_x","end_y"]].reset_index(drop = True)
        elapsed = timeit.default_timer() - start_time
        #print(f"2.5:{elapsed}")
        start_time = timeit.default_timer()#this is really slow...
        input_df_act["end_x"] = coord_x / x_lim * 105 + 105 / x_lim / 2
        input_df_act["end_y"] = coord_y / y_lim * 68 + 68 / y_lim / 2
        #metrics = self.rate_optimized(game, action, pass_selection_surface, pass_success_surface, pass_value_surface_offensive_success, pass_value_surface_defensive_success,pass_value_surface_offensive_fail, pass_value_surface_defensive_fail, db, game_df_actions)
        elapsed = timeit.default_timer() - start_time
        #print(f"3:{elapsed}")
        start_time = timeit.default_timer()
        metrics = rate(input_df_act, game, action, pass_selection_surface, pass_success_surface, pass_value_surface_offensive_success, pass_value_surface_defensive_success,pass_value_surface_offensive_fail, pass_value_surface_defensive_fail, db, game_df_actions)
        #print(metrics.columns) 
        #print(true_ends)               
        metrics["Dist_From_True"] = (metrics["end_x"] - true_ends[0])**2 + (metrics["end_y"] - true_ends[1])**2
        closest_idx = np.where(metrics["Dist_From_True"] == min(metrics["Dist_From_True"]))[0]#getting closest pass to actual pass and then replacing that one with the original pass
        metrics = metrics.reset_index(drop = True)#is the slowest here?
        cols = ["end_x", "end_y", "selection_probability", "success_probability", "value_success", "value_fail"]
        metrics["True_Location"] = 0
        elapsed = timeit.default_timer() - start_time
        #print(f"4:{elapsed}")
        start_time = timeit.default_timer()
        for col in cols:
            metrics[col][closest_idx[0]] = game_ogs[col].loc[action]
        metrics["selection_probability"] = np.float64(metrics["selection_probability"])
        metrics["True_Location"][closest_idx[0]] = 1
        metrics["expected_utility"] = (metrics["success_probability"] * metrics["value_success"]) + ((1 - metrics["success_probability"]) * metrics["value_fail"])
        metrics["evaluation_criterion"] = metrics["expected_utility"] - sum(metrics["selection_probability"] * metrics["expected_utility"])
        metrics["selection_criterion"] = sum(
        metrics["selection_probability"] * (metrics["evaluation_criterion"])**2
        )
        metrics = metrics.drop(columns = ["Dist_From_True"])
        #if summarize:
        #    metrics = metrics[metrics["True_Location"] == 1]
        #    metrics = metrics[["original_event_id", "game_id", "action_id", "start_x","start_y", "end_x", "end_y", "result_id", "selection_criterion", "evaluation_criterion"]]
        #if custom_pass is not None:
        #    continue
        alldf.append(metrics)
    
combined = pd.concat(alldf)

DFL-MAT-J03YKP 0
0.0038070519985922147
DFL-MAT-J03YKP 2
0.003767837999475887
DFL-MAT-J03YKP 5
0.003806191001785919
DFL-MAT-J03YKP 9
0.0037979849985276815
DFL-MAT-J03YKP 10
0.003858930998831056
DFL-MAT-J03YKP 12
0.003843872000288684
DFL-MAT-J03YKP 14
0.0037985870003467426
DFL-MAT-J03YKP 16
0.0038177330025064293
DFL-MAT-J03YKP 17
0.0036757449997821823
DFL-MAT-J03YKP 19
0.003861635999783175
DFL-MAT-J03YKP 20
0.0038203369986149482
DFL-MAT-J03YKP 21
0.003765123001358006
DFL-MAT-J03YKP 23
0.0038022040025680326
DFL-MAT-J03YKP 24
0.004614805002347566
DFL-MAT-J03YKP 25
0.003911028998118127
DFL-MAT-J03YKP 26
0.005032140998082468
DFL-MAT-J03YKP 28
0.0038452650005638134
DFL-MAT-J03YKP 29
0.003871814002195606
DFL-MAT-J03YKP 30
0.003864009999233531
DFL-MAT-J03YKP 32
0.00386079400050221
DFL-MAT-J03YKP 33
0.0038289030017040204
DFL-MAT-J03YKP 38
0.0038676770018355455
DFL-MAT-J03YKP 39
0.003872355999192223
DFL-MAT-J03YKP 41
0.003975829997216351
DFL-MAT-J03YKP 43
0.003797504999965895
DFL-MAT-J03YKP 45
0.